# Selección de targets MOP con cobertura Rubin

Este notebook selecciona targets visibles de MOP, consulta su cobertura en el DataRelease de Rubin elegido y genera tablas, mapas y reportes individuales. Ejecutar las secciones en orden.


## 1. Configuración y conexiones

Elegí `DATA_RELEASE_NAME` entre `DP0.1`, `DP0.2`, `DP1` y `DP2`. El perfil configura servicio RSP, repositorio/collections Butler, tabla y columnas TAP, y datasets de coadd; `DP2` conserva exactamente la corrida actual.


In [ ]:
from functools import partial
from importlib import reload
from pathlib import Path

import pandas as pd
from lsst.daf.butler import Butler
from lsst.rsp import RSPDiscovery
import mop_api.client as mop_client
import target_selection_pipeline as pipeline
import data_release_config as release_config
import photometry as photometry_tools
import target_report as report_tools

# Recarga cambios locales al volver a ejecutar el notebook en el mismo kernel.
reload(mop_client)
reload(release_config)
reload(pipeline)
reload(photometry_tools)
reload(report_tools)
MOPClient = mop_client.MOPClient
run_pipeline = pipeline.run_pipeline
get_data_release = release_config.get_data_release

DATA_RELEASE_NAME = "DP2"  # DP0.1, DP0.2, DP1 o DP2
DATA_RELEASE = get_data_release(DATA_RELEASE_NAME)

START_DATE = "2026-08-01"
END_DATE = "2026-08-15"
OUTPUT_DIR = Path("outputs")
OBSERVATORY = "El Leoncito"

butler_options = {}
if DATA_RELEASE.butler_collections is not None:
    butler_options["collections"] = DATA_RELEASE.butler_collections
butler = Butler(DATA_RELEASE.butler_repo, **butler_options)
discovery = RSPDiscovery(DATA_RELEASE.rsp_instance)
tap_service = discovery.get_tap_client()
mop = MOPClient()

if not hasattr(mop, "microlensing_parameters"):
    raise RuntimeError(
        "La instalación de mop_api no incluye parámetros tabulares. "
        "Actualizá mop_api desde la rama publicada antes de continuar."
    )


## 2. Configurar los reportes individuales

Las funciones viven en los módulos instalados; aquí sólo se conectan con MOP, Butler y el Data Release elegido.


In [ ]:
PHOTOMETRY_CACHE_DIR = OUTPUT_DIR / "photometry"

# Se inyectan conexiones y configuración; la implementación vive en los módulos.
load_event_photometry = partial(
    photometry_tools.load_event_photometry,
    mop=mop,
    cache_dir=PHOTOMETRY_CACHE_DIR,
)
plot_target = partial(
    report_tools.plot_target,
    butler=butler,
    tap_service=tap_service,
    data_release=DATA_RELEASE,
)


## 3. Ejecutar el pipeline

Cada página MOP se descarga una sola vez con concurrencia limitada y produce parámetros más fotometría. Los CSV, páginas MOP, 404, fotometrías, consultas de cobertura Rubin y plots quedan cacheados; la celda informa el progreso por etapas. Usá `reuse_cache=False` sólo para forzar datos nuevos.


In [ ]:
combined, paths = run_pipeline(
    mop=mop,
    tap_service=tap_service,
    start_date=START_DATE,
    end_date=END_DATE,
    root_dir=OUTPUT_DIR,
    observatory=OBSERVATORY,
    target_plotter=lambda row, coverage: plot_target(
        row, calexps=coverage, photometry=load_event_photometry(row)
    ),
    max_workers=4,
    reuse_cache=True,
    overwrite_target_plots=False,
    verbose=True,
    data_release=DATA_RELEASE,
)

print(f"Targets procesados: {len(combined)}")
print(f"Productos guardados en: {paths['run'].resolve()}")
combined.head()


## 4. Explorar los resultados

Las siguientes celdas muestran los productos principales sin necesidad de recorrer manualmente las carpetas.


In [ ]:
from IPython.display import Image, Markdown, display
target_summary = pd.read_csv(paths["tables"] / "target_summary.csv")
n_visible = len(target_summary)
n_coverage = int(target_summary["matched_release"].fillna(False).sum())
n_photometry = int(target_summary["mop_photometry_points"].fillna(0).gt(0).sum())
n_both = int(target_summary["matched_with_photometry"].fillna(False).sum())
display(Markdown(
    f"### Resumen de la corrida\n"
    f"- **DataRelease:** {DATA_RELEASE.name}\n"
    f"- **Targets visibles:** {n_visible}\n"
    f"- **Con cobertura {DATA_RELEASE.name}:** {n_coverage}\n"
    f"- **Con fotometría MOP:** {n_photometry}\n"
    f"- **Con cobertura y fotometría:** {n_both}\n"
    f"- **Directorio:** {paths['run'].resolve()}"
))
summary_columns = [
    "Target", "priority", "release_n_visits",
    *[column for column in target_summary if column.startswith("n_visits_")],
    "mop_photometry_points", "t_E_days", "t_0_HJD", "u_0",
]
display(target_summary[[column for column in summary_columns if column in target_summary]].head(20))


### Tabla resumen completa

El PNG contiene todos los targets, las visitas por filtro y los principales parámetros MOP. El CSV equivalente conserva los valores sin abreviar.


In [ ]:
summary_png = paths["tables"] / "target_summary.png"
display(Image(filename=str(summary_png)))
print(f"CSV completo: {(paths['tables'] / 'target_summary.csv').resolve()}")


### Mapas del cielo


In [ ]:
map_paths = [
    paths["sky_plots"] / "sky_by_mag_and_visits.png",
    paths["sky_plots"] / "sky_bulge_zoom_mag_and_visits.png",
]
for map_path in map_paths:
    if map_path.exists():
        display(Markdown(f"#### {map_path.stem.replace('_', ' ')}"))
        display(Image(filename=str(map_path), width=1100))


### Ejemplos de reportes individuales

Por defecto se muestran solamente tres para no sobrecargar el notebook. Cambiá MAX_REPORTS_TO_DISPLAY si necesitás ver más.


In [ ]:
MAX_REPORTS_TO_DISPLAY = 3
report_paths = sorted(paths["targets"].glob("*_target_report.png"))
print(f"Reportes disponibles: {len(report_paths)}")
for report_path in report_paths[:MAX_REPORTS_TO_DISPLAY]:
    target_name = report_path.name.removesuffix("_target_report.png")
    display(Markdown(f"#### {target_name}"))
    display(Image(filename=str(report_path), width=1300))


## 5. Validación técnica

Comprueba que los parámetros MOP estén presentes en las tablas y muestra dónde encontrar los productos.


In [ ]:
summary_path = paths["tables"] / "visible_summary.csv"
combined_path = paths["tables"] / "combined_targets.csv"
summary_csv = pd.read_csv(summary_path)
combined_csv = pd.read_csv(combined_path)
parameter_columns = [
    column for column in combined.columns
    if column.startswith("mop_")
    and column not in {"mop_link", "mop_parameters_status", "mop_parameters_error"}
]
missing_summary = sorted(set(parameter_columns) - set(summary_csv.columns))
missing_combined = sorted(set(parameter_columns) - set(combined_csv.columns))
if missing_summary or missing_combined:
    raise RuntimeError(f"Parámetros ausentes en CSV: summary={missing_summary}, combined={missing_combined}")
status_counts = (
    combined["mop_parameters_status"].value_counts(dropna=False).to_dict()
    if "mop_parameters_status" in combined else {}
)
print(f"Estado de páginas MOP: {status_counts}")
print(f"Parámetros medidos: {parameter_columns or 'ninguno'}")
print(f"Reportes con coadd: {len(report_paths)}")
print(f"Tabla final: {combined_path.resolve()}")
print(f"Resumen visual: {summary_png.resolve()}")
